In [ ]:
pip install ultralytics

In [ ]:
from pathlib import Path
import random
import shutil
import xml.etree.ElementTree as ET

import kagglehub
import matplotlib.pyplot as plt
import pandas as pd
import yaml
from ultralytics import YOLO

In [ ]:
# Configuration

DATASET_NAME = "andrewmvd/car-plate-detection"
WORKING_DIR = Path("/kaggle/working")
RANDOM_SEED = 42
TRAIN_RATIO = 0.75
CLASS_ID = 0
CLASS_NAME = "license"

downloaded_data_dir = Path(kagglehub.dataset_download(DATASET_NAME))
print(f"Downloaded Kaggle dataset to: {downloaded_data_dir}")

downloaded_images_dir = downloaded_data_dir / "images"
downloaded_annotations_dir = downloaded_data_dir / "annotations"

images_dir = WORKING_DIR / "images"
labels_dir = WORKING_DIR / "labels"

train_images_dir = images_dir / "train"
val_images_dir = images_dir / "val"
train_labels_dir = labels_dir / "train"
val_labels_dir = labels_dir / "val"

In [ ]:
def convert_min_max_to_center_dims(width: float, height: float, xmin: float, xmax: float, ymin: float, ymax: float):
    """Convert Pascal VOC bounding box coordinates to YOLO format.

    Returns:
        (x_center, y_center, box_width, box_height) normalized to image size.
    """
    x_center = ((xmin + xmax) / 2) / width
    y_center = ((ymin + ymax) / 2) / height
    box_width = (xmax - xmin) / width
    box_height = (ymax - ymin) / height
    return x_center, y_center, box_width, box_height


def parse_xml_to_yolo_lines(xml_path: Path, class_id: int = 0) -> list[str]:
    """Read one Pascal VOC XML annotation file and return YOLO label lines."""
    if not xml_path.exists():
        return []

    tree = ET.parse(xml_path)
    root = tree.getroot()

    width = float(root.find("size/width").text)
    height = float(root.find("size/height").text)

    lines = []
    for obj in root.findall("object"):
        xmin = float(obj.find("bndbox/xmin").text)
        xmax = float(obj.find("bndbox/xmax").text)
        ymin = float(obj.find("bndbox/ymin").text)
        ymax = float(obj.find("bndbox/ymax").text)

        x_center, y_center, box_width, box_height = convert_min_max_to_center_dims(
            width, height, xmin, xmax, ymin, ymax
        )

        lines.append(f"{class_id} {x_center} {y_center} {box_width} {box_height}")

    return lines

In [ ]:
def reset_output_directories() -> None:
    """Delete and recreate the working dataset folders."""
    if WORKING_DIR.exists():
        shutil.rmtree(WORKING_DIR)

    for folder in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
        folder.mkdir(parents=True, exist_ok=True)


def split_image_files(images_path: Path, train_ratio: float = 0.75, seed: int = 42) -> tuple[list[str], list[str]]:
    """Split image file names into train and validation sets."""
    if not 0 < train_ratio < 1:
        raise ValueError("train_ratio must be between 0 and 1.")

    image_files = [
        file_name
        for file_name in images_path.iterdir()
        if file_name.suffix.lower() in {".png", ".jpg", ".jpeg"}
    ]

    image_names = [file.name for file in image_files]
    random.Random(seed).shuffle(image_names)

    split_index = int(len(image_names) * train_ratio)
    return image_names[:split_index], image_names[split_index:]


def process_split(file_names: list[str], target_images_dir: Path, target_labels_dir: Path) -> None:
    """Copy images and generate matching YOLO label files for one dataset split."""
    for image_name in file_names:
        base_name = Path(image_name).stem
        xml_name = f"{base_name}.xml"

        source_image_path = downloaded_images_dir / image_name
        source_xml_path = downloaded_annotations_dir / xml_name

        target_image_path = target_images_dir / image_name
        target_label_path = target_labels_dir / f"{base_name}.txt"

        shutil.copy2(source_image_path, target_image_path)

        yolo_lines = parse_xml_to_yolo_lines(source_xml_path, class_id=CLASS_ID)
        target_label_path.write_text("\n".join(yolo_lines))


def prepare_yolo_dataset(train_ratio: float = 0.75, seed: int = 42) -> None:
    """Prepare train/validation image and label folders for YOLO training."""
    reset_output_directories()
    train_files, val_files = split_image_files(downloaded_images_dir, train_ratio=train_ratio, seed=seed)

    process_split(train_files, train_images_dir, train_labels_dir)
    process_split(val_files, val_images_dir, val_labels_dir)

    print("Dataset prepared successfully.")
    print(f"Train images: {len(train_files)}")
    print(f"Validation images: {len(val_files)}")

In [ ]:
def create_yolo_data_yaml(output_dir: Path) -> Path:
    """Create the YOLO data.yaml configuration file."""
    data = {
        "path": str(WORKING_DIR),
        "train": "images/train",
        "val": "images/val",
        "names": {CLASS_ID: CLASS_NAME},
    }

    yaml_path = output_dir / "data.yaml"
    with open(yaml_path, "w") as file:
        yaml.dump(data, file, sort_keys=False)

    print(f"YOLO config file created at: {yaml_path}")
    return yaml_path

In [ ]:
prepare_yolo_dataset(train_ratio=TRAIN_RATIO, seed=RANDOM_SEED)
config_path = create_yolo_data_yaml(WORKING_DIR)

In [ ]:
# yolov8n.yaml for training from scratch
# yolov8n.pt for using pretrained weights

model = YOLO("yolov8n.yaml")

In [ ]:
model.train(
    data=str(config_path),
    epochs=400,
    batch=32,
    optimizer="SGD",
    lr0=0.001,
    patience=20,
)

In [ ]:
def plot_training_results(results_csv_path: str = "runs/detect/train/results.csv") -> None:
    """Plot training and validation box loss from YOLO results."""
    results_path = Path(results_csv_path)
    if not results_path.exists():
        raise FileNotFoundError(f"Could not find results file: {results_path}")

    results = pd.read_csv(results_path)

    plt.figure(figsize=(10, 5))
    plt.plot(results["epoch"], results["train/box_loss"], label="train")
    plt.plot(results["epoch"], results["val/box_loss"], label="val")
    plt.xlabel("Epoch")
    plt.ylabel("Box Loss")
    plt.title("YOLOv8 Training Performance")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


plot_training_results()

In [ ]:
# new_model = YOLO('/content/last.pt')

In [ ]:
# new_model.train(data=config_path, epochs=100)